In [5]:
# =========================================
# Fetch Current Forecast from Open-Meteo 
# =========================================

# Libraries
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import statistics 
import math
import requests 
from pathlib import Path 
from datetime import datetime, timezone  

# -----------------------------------------
# 1. Configuration
# -----------------------------------------

LOCATION = "London"
LAT = 51.505 
LON = 0.05500000000000002 
SENSOR_STATION = "EGLC"
URL = "https://api.open-meteo.com/v1/forecast" 

model = "ecmwf_ifs025"

params = {
    "latitude": LAT, 
    "longitude": LON, 
    "hourly": "temperature_2m",
    "forecast_days": 3,
    "models": model,
    "timezone": "Europe/London"
}



# -----------------------------
# 2.  Download Forecast
# -----------------------------
response = requests.get(URL, params=params)
response.raise_for_status()

data = response.json()



# -----------------------------
# 3. Convert to DataFrame
# -----------------------------
df = pd.DataFrame({
    "forecast_time": pd.to_datetime(data["hourly"]["time"], utc=True),
    "temperature": data["hourly"]["temperature_2m"]
})

# Metadata
df["location"] = LOCATION
df["latitude"] = LAT
df["longitude"] = LON
df["model"] = params["models"]   # Placeholder for now
df["fetched_at"] = datetime.now(timezone.utc)

# Horizon (hours from fetch)
df["horizon_hours"] = (
    df["forecast_time"] - df["fetched_at"]
).dt.total_seconds() / 3600



# -----------------------------
# 4. Save
# -----------------------------
csv_file = Path(f"{SENSOR_STATION}_forecast_current.csv")
df.to_csv(csv_file, index=False)

print(df.head())

              forecast_time  temperature location  latitude  longitude  \
0 2026-06-28 00:00:00+00:00         22.8   London    51.505      0.055   
1 2026-06-28 01:00:00+00:00         21.9   London    51.505      0.055   
2 2026-06-28 02:00:00+00:00         21.3   London    51.505      0.055   
3 2026-06-28 03:00:00+00:00         20.8   London    51.505      0.055   
4 2026-06-28 04:00:00+00:00         20.5   London    51.505      0.055   

          model                       fetched_at  horizon_hours  
0  ecmwf_ifs025 2026-06-28 20:55:20.855080+00:00      -20.92246  
1  ecmwf_ifs025 2026-06-28 20:55:20.855080+00:00      -19.92246  
2  ecmwf_ifs025 2026-06-28 20:55:20.855080+00:00      -18.92246  
3  ecmwf_ifs025 2026-06-28 20:55:20.855080+00:00      -17.92246  
4  ecmwf_ifs025 2026-06-28 20:55:20.855080+00:00      -16.92246  
